[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/intronlp-2026/blob/main/notebooks/week-03.ipynb)

# 3주차 실습: 바이그램 모델 만들기와 문장 생성

**목표.** 여덟 문장에서 이웃한 두 단어 짝을 세어 확률표를 만들고, 그 표를 따라 걸어 문장을 생성한다.

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 짝 세기

강의 노트의 코드 그대로입니다. 출력은 `Counter({'고기국수': 2, '흑돼지': 1, '감귤': 1})` 입니다. 강의 1에서 손으로 만든 `것은` 표와 같은지 확인하세요.

In [1]:
sentences = [
    "혼저 옵서예 제주",
    "혼저 옵서예 한라산",
    "제주 바다는 넓다",
    "우도 바다는 맑다",
    "제주에서 먹고 싶은 것은 고기국수",
    "오늘 먹고 싶은 것은 고기국수",
    "제주에서 먹고 싶은 것은 흑돼지",
    "제주에서 사고 싶은 것은 감귤",
]

from collections import Counter, defaultdict

pairs = defaultdict(Counter)
for s in sentences:
    words = ["[시작]"] + s.split() + ["[끝]"]
    for a, b in zip(words, words[1:]):
        pairs[a][b] += 1

print(pairs["것은"])

Counter({'고기국수': 2, '흑돼지': 1, '감귤': 1})


### 1-2. 활동 A의 네 문장도 같은 코드로

말뭉치만 바꿨고 나머지는 위와 같습니다. 출력은 아래와 같아야 합니다.

```
오늘 Counter({'오름에': 2, '바다에': 1})
바다에 Counter({'간다': 2})
오름에 Counter({'간다': 1, '오른다': 1})
```

In [2]:
activity = ["오늘 바다에 간다", "오늘 오름에 간다", "내일 바다에 간다", "오늘 오름에 오른다"]

pairs_a = defaultdict(Counter)
for s in activity:
    words = ["[시작]"] + s.split() + ["[끝]"]
    for a, b in zip(words, words[1:]):
        pairs_a[a][b] += 1

for w in ["오늘", "바다에", "오름에"]:
    print(w, pairs_a[w])

오늘 Counter({'오름에': 2, '바다에': 1})
바다에 Counter({'간다': 2})
오름에 Counter({'간다': 1, '오른다': 1})


### 1-3. 횟수를 확률로

한 줄의 횟수를 그 줄의 합으로 나누면 확률이 됩니다. `[시작]` 줄은 제주에서 0.38, 혼저 0.25, 제주 0.13, 우도 0.13, 오늘 0.13 입니다. 소수 둘째 자리에서 반올림했기 때문에 더하면 1을 살짝 넘을 수 있습니다.

In [3]:
def to_prob(counter):
    total = sum(counter.values())
    return {w: int(c / total * 100 + 0.5) / 100 for w, c in counter.most_common()}

print(to_prob(pairs["[시작]"]))

{'제주에서': 0.38, '혼저': 0.25, '제주': 0.13, '우도': 0.13, '오늘': 0.13}


### 1-4. 표를 따라 걸어 문장 만들기

`[시작]` 에서 출발해 표에서 한 칸씩 뽑아 `[끝]` 에 닿으면 문장 하나가 됩니다. 셀을 다시 실행할 때마다 다른 문장이 나올 수 있습니다.

In [4]:
import random

def generate():
    word, out = "[시작]", []
    while True:
        nexts = pairs[word]
        word = random.choices(list(nexts), weights=list(nexts.values()))[0]
        if word == "[끝]":
            return " ".join(out)
        out.append(word)

for _ in range(5):
    print(generate())

제주에서 먹고 싶은 것은 흑돼지
혼저 옵서예 한라산
혼저 옵서예 제주 바다는 맑다
제주 바다는 넓다
제주에서 먹고 싶은 것은 감귤


어색한 문장이 나오면 아래 세 갈래 중 어디에 해당하는지 생각해 봅니다.

1. **본 적 없는 말은 절대 안 나온다.** 갈치조림은 아무리 돌려도 나오지 않는다
2. **바로 앞 한 단어밖에 기억을 못 한다.** `사고 싶은 것은 고기국수` 같은 문장이 나올 수 있다
3. **센 것이 너무 적다.** 대부분의 칸이 1.00이라 정해진 길만 따라간다

## 2. 한 지점만 바꿔 보기

아래 셀의 `TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.

`word` 를 다른 앞말로 바꿔 그 줄의 횟수와 확률을 봅니다. 예: `"제주에서"`, `"바다는"`, `"[시작]"`.

말뭉치에 없는 말을 넣으면 오류 없이 빈 `Counter()` 와 빈 표가 나옵니다. 오류가 아니라 **그 말이 말뭉치에 없었다는 신호**입니다.

In [5]:
# TODO: 따옴표 안의 앞말을 바꿔 보세요
word = "제주에서"

# 아래는 그대로 둡니다
print("횟수:", pairs[word])
print("확률:", to_prob(pairs[word]) if pairs[word] else "{} (말뭉치에 없는 앞말)")

횟수: Counter({'먹고': 2, '사고': 1})
확률: {'먹고': 0.67, '사고': 0.33}


## 3. 확인 질문

1. 1-1의 `것은` 표를 그대로 옮겨 적고, 손으로 센 표와 같았는지 적으세요.
2. 1-4에서 생성한 문장 세 개 이상을 옮겨 적으세요.
3. 그중 어색한 문장 하나를 골라 위 세 갈래 중 어디에 해당하는지와 그 이유를 적으세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

**답**

1. `것은` 표: 고기국수 2, 흑돼지 1, 감귤 1. 강의 노트의 표와 같았다.
2. 생성한 문장:
   - 제주에서 먹고 싶은 것은 흑돼지
   - 혼저 옵서예 한라산
   - 혼저 옵서예 제주 바다는 맑다
   - 제주 바다는 넓다
   - 제주에서 먹고 싶은 것은 감귤
3. "혼저 옵서예 제주 바다는 맑다"가 어색하다. 세 갈래 중 2번(바로 앞 한 단어밖에 기억을 못 한다)이다. "혼저 옵서예 제주"와 "우도 바다는 맑다"가 섞인 것인데, 모델은 바로 앞 단어 "제주"만 보고 그 뒤에 나온 적 있는 "바다는"을 이었기 때문이다.


## 4. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/intronlp-2026)의 `assignments/week-03/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.